In [2]:
import pandas as pd
import joblib
from whatsapp import WhatsappFilrt

# 1. Instantiate class and load chat data
wf = WhatsappFilrt()
df = wf.PreProcessing("Chat.txt")

# Clean text and name columns
df["Text"] = df["Text"].fillna("").astype(str)
df["Name"] = df["Name"].fillna("Unknown").astype(str).str.lower().str.strip()
df = df[df["Name"] != "unknown"].copy()


# =========================================================
# 1. ROW-BY-ROW ML FLIRT PREDICTION (USING TRAINED MODEL)
# =========================================================
print("=" * 50)
print("1. ML FLIRT CLASSIFICATION")
print("=" * 50)

try:
    flirt_model = joblib.load("best_whatsapp_classifier.pkl")
    print("Model 'best_whatsapp_classifier.pkl' loaded successfully.")
except FileNotFoundError:
    print("Error: 'best_whatsapp_classifier.pkl' not found. Run Step 1 first.")
    exit()

# Predict Flirt Intent row-by-row
df["Predicted_Flirt_Label"] = flirt_model.predict(df["Text"].str.lower())

# Extract probability/confidence scores per row
probs = flirt_model.predict_proba(df["Text"].str.lower())
df["Flirt_Confidence_%"] = (probs.max(axis=1) * 100).round(2)

df["Is_Flirt"] = df["Predicted_Flirt_Label"] == "Flirt"

# Flirt Summary by Participant
flirt_summary = df.groupby("Name").agg(
    Total_Messages=("Text", "count"),
    Predicted_Flirt_Messages=("Is_Flirt", "sum")
).reset_index()

flirt_summary["Flirt_%"] = round(
    (flirt_summary["Predicted_Flirt_Messages"] / flirt_summary["Total_Messages"]) * 100, 2
)
print(flirt_summary.to_string(index=False))

# Show Row-by-Row Predicted Flirt Detections
flirty_rows = df[df["Is_Flirt"] == True][["Date", "Time", "Name", "Text", "Flirt_Confidence_%"]]
print(f"\nTotal Flirty Rows Predicted: {len(flirty_rows)}")
if not flirty_rows.empty:
    print(flirty_rows.head(10).to_string(index=False))
print("\n")


# =========================================================
# 2. TALKATIVE & LESS TALKATIVE PARTICIPANTS
# =========================================================
print("=" * 50)
print("2. TALKATIVE METRICS")
print("=" * 50)

message_counts = df["Name"].value_counts()
most_talkative = message_counts.idxmax().upper()
least_talkative = message_counts.idxmin().upper()

print(f"• Most Talkative Person : {most_talkative} ({message_counts.max()} messages)")
print(f"• Less Talkative Person : {least_talkative} ({message_counts.min()} messages)\n")
print(message_counts.to_string())
print("\n")


# =========================================================
# 3. MOST ACTIVE DAY & MOST ACTIVE TIME
# =========================================================
print("=" * 50)
print("3. ACTIVITY TIMINGS")
print("=" * 50)

df["Datetime"] = pd.to_datetime(df["Date"] + " " + df["Time"], errors="coerce")
df["Day_of_Week"] = df["Datetime"].dt.day_name()
df["Hour"] = df["Datetime"].dt.hour

most_active_day = df["Day_of_Week"].mode()[0]
day_counts = df["Day_of_Week"].value_counts()[most_active_day]

most_active_hour = df["Hour"].mode()[0]
time_slot = f"{int(most_active_hour):02d}:00 - {int(most_active_hour)+1:02d}:00"

print(f"• Most Active Day  : {most_active_day} ({day_counts} messages)")
print(f"• Peak Active Time : {time_slot}")
print("\n")


# =========================================================
# 4. MEDIA COUNT SENT BY EACH PERSON
# =========================================================
print("=" * 50)
print("4. MEDIA COUNTS")
print("=" * 50)

media_mask = df["Text"].str.contains("<Media omitted>|media omitted|photo|video|sticker", case=False, na=False)
df["Is_Media"] = media_mask

media_summary = df.groupby("Name")["Is_Media"].sum().reset_index(name="Media_Count")
print(media_summary.to_string(index=False))
print("\n")


# =========================================================
# 5. MISSED CALLS (ALL PARTICIPANTS)
# =========================================================
print("=" * 50)
print("5. MISSED CALLS DETECTED")
print("=" * 50)

missed_call_mask = df["Text"].str.contains("missed voice call|missed video call", case=False, na=False)
df["Is_Missed_Call"] = missed_call_mask

missed_calls_df = df[df["Is_Missed_Call"] == True][["Date", "Time", "Name", "Text"]]

print(f"Total Missed Calls Logged: {len(missed_calls_df)}\n")
if not missed_calls_df.empty:
    print(missed_calls_df.to_string(index=False))

1. ML FLIRT CLASSIFICATION
Model 'best_whatsapp_classifier.pkl' loaded successfully.
                                                                                                         Name  Total_Messages  Predicted_Flirt_Messages  Flirt_%
                                                                                         +91 63832 75541 left               1                         0     0.00
                                                                                              +91 70101 48941               6                         0     0.00
                                                                                              +91 78718 33139               7                         1    14.29
                                                                        +91 78718 33139 added +91 96551 11513               1                         0     0.00
                                                                                         +91 78718 33139 left 

C:\Users\Diwali 6\AppData\Local\Temp\ipykernel_14440\1271191868.py:81: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Datetime"] = pd.to_datetime(df["Date"] + " " + df["Time"], errors="coerce")


• Most Active Day  : Monday (350 messages)
• Peak Active Time : 17:00 - 18:00


4. MEDIA COUNTS
                                                                                                         Name  Media_Count
                                                                                         +91 63832 75541 left            0
                                                                                              +91 70101 48941            1
                                                                                              +91 78718 33139            2
                                                                        +91 78718 33139 added +91 96551 11513            0
                                                                                         +91 78718 33139 left            0
                                                                                              +91 79072 88212            0
                                           